# Random Forest Regression

training random forest regression in blocks using baseline features + traficom features

## 1. Imports

This notebook mirrors the linear-regression workflow but swaps the estimator for `RandomForestRegressor` while keeping the same train/validation split and feature-set comparisons.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


## 2. Load data

In [ ]:
train_path = "../../datasets/splits/train_grouped.csv"
validation_path = "../../datasets/splits/validation_grouped.csv"

In [ ]:
train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)

## 3. Quick data checks

In [ ]:
print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

In [ ]:
print("Train info")
train_df.info()

print("\nValidation info")
validation_df.info()

## 4. Define feature groups

Registry lifecycle features describe vehicle registration-history patterns from Traficom data. Listing dynamics features describe scrape-window behavior of listings.

In [ ]:
baseline_features = [
    "part_name",
    "quality_grade",
    "oem_number",
    "mileage",
    "brand",
    "model",
    "category",
    "subcategory",
    "year_start",
    "year_end",
    "year_span",
    "year_mid",
    "repair_status",
]

traficom_features = [
    "model_total_registered",
    "model_median_vehicle_age",
    "model_mean_vehicle_age",
    "model_median_mileage",
    "model_mean_mileage",
    "model_median_engine_cc",
    "model_median_power_kw",
    "model_median_mass_kg",
    "brand_total_registered",
    "brand_median_vehicle_age",
    "brand_mean_vehicle_age",
    "brand_median_mileage",
    "brand_mean_mileage",
]

# Registry lifecycle features describe vehicle registration-history features.
registry_lifecycle_candidates = [
    "model_firstreg_total_2014_2026",
    "model_firstreg_recent_share",
    "model_firstreg_old_share",
    "model_firstreg_weighted_year",
    "model_firstreg_peak_year",
    "model_firstreg_peak_count",
    "model_firstreg_year_span",
    "brand_firstreg_total_2014_2026",
    "brand_firstreg_recent_share",
    "brand_firstreg_old_share",
    "brand_firstreg_weighted_year",
    "brand_firstreg_peak_year",
    "brand_firstreg_peak_count",
    "brand_firstreg_year_span",
]

registry_lifecycle_features = [
    column for column in registry_lifecycle_candidates 
    if column in train_df.columns

]

missing_registry_lifecycle_features = [
    column for column in registry_lifecycle_candidates 
    if column not in train_df.columns
]

traficom_extended_candidates = [
    "model_share_of_market",
    "model_share_within_brand",
    "model_share_over_10y",
    "model_share_over_200k_km",
    "model_automatic_share",
    "model_petrol_share",
    "model_diesel_share",
    "model_ev_share",
    "model_hybrid_share",
    "model_turbo_share",
    "brand_median_engine_cc",
    "brand_median_power_kw",
    "brand_median_mass_kg",
    "brand_share_of_market",
    "brand_share_over_10y",
    "brand_share_over_200k_km",
    "brand_automatic_share",
    "brand_petrol_share",
    "brand_diesel_share",
    "brand_ev_share",
    "brand_hybrid_share",
    "brand_turbo_share",
]

traficom_extended_features = [
    column for column in traficom_extended_candidates if column in train_df.columns
]

missing_traficom_extended_features = [
    column for column in traficom_extended_candidates if column not in train_df.columns
]

print("Found registry lifecycle features:")
print(registry_lifecycle_features)

print("\nMissing registry lifecycle features:")
print(missing_registry_lifecycle_features)

print("\nFound extended Traficom features:")
print(traficom_extended_features)

print("\nMissing extended Traficom features:")
print(missing_traficom_extended_features)

# Listing dynamics features describe scrape-window listing behavior,
# not registry lifecycle or vehicle registration-history features.
listing_dynamics_features = [
    "times_observed",
    "observed_span_days",
    "price_changed_flag",
    "price_change_count",
    "absolute_price_change",
    "relative_price_change_pct",
]

# Keep this alias so the existing notebook cells continue to run.
lifecycle_features = listing_dynamics_features

recommended_inclusion_reasons = {
    "first_seen_date": "Observed listing start within the crawl window; improved validation when included.",
    "last_seen_date": "Observed listing end within the crawl window; improved validation when included.",
    "brand_is_known_model_family": "Low-cardinality metadata flag; slight validation improvement when included.",
    "mileage_missing_flag": "Tracks rows where mileage was originally missing before hierarchical median imputation.",
}

recommended_exclusion_reasons = {
    "product_id": "High-cardinality listing identifier; encourages memorization and hurt MAE.",
    "scrape_date": "Current crawl wave rather than part value; worsened validation when included.",
    "brand_merge_key": "Redundant normalized brand key that overlaps with brand.",
    "model_merge_key": "Redundant normalized model key that overlaps with model.",
    "model_family_clean": "Collapsed model family duplicate of the existing model labels.",
    "model_looks_like_part_taxonomy": "Constant in the training split, so it adds no signal.",
}

manual_all_feature_groups = list(dict.fromkeys(
    baseline_features
    + traficom_features
    + registry_lifecycle_features
    + traficom_extended_features
    + listing_dynamics_features
))

recommended_excluded_features = set(recommended_exclusion_reasons)

recommended_model_features = [
    column for column in train_df.columns
    if column != "price" and column not in recommended_excluded_features
]

missing_from_previous_manual_all = [
    column for column in recommended_model_features if column not in manual_all_feature_groups
]

feature_audit_df = pd.DataFrame(
    [
        {
            "feature": column,
            "status": "missing_from_previous_manual_all",
            "reason": recommended_inclusion_reasons.get(
                column,
                "New candidate feature found in the dataset and included in the recommended model feature set.",
            ),
        }
        for column in missing_from_previous_manual_all
    ]
    + [
        {
            "feature": column,
            "status": "recommended_exclusion",
            "reason": recommended_exclusion_reasons[column],
        }
        for column in sorted(recommended_excluded_features)
    ]
)

print("Columns missing from the previous manual all-features set:")
print(missing_from_previous_manual_all)

print("\nRecommended exclusions:")
print(sorted(recommended_excluded_features))

print("\nNumber of recommended model features:", len(recommended_model_features))

feature_audit_df

## 5. Select baseline feature set

In [ ]:
features_baseline_only = baseline_features

assert len(features_baseline_only) > 0, "Add at least one column to baseline_features before training."

print("Number of baseline features:", len(features_baseline_only))
features_baseline_only

## 6. Build X and y

In [ ]:
missing_features = [column for column in features_baseline_only if column not in train_df.columns]
assert len(missing_features) == 0, f"These selected features are missing from the dataset: {missing_features}"

X_train = train_df[features_baseline_only].copy()
X_validation = validation_df[features_baseline_only].copy()

y_train = train_df["price"].copy()
y_validation = validation_df["price"].copy()

## 7. Log-transform target

In [ ]:
y_train_log = np.log(y_train)

y_train_log.head()

## 8. Detect column types

In [ ]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_features

## 8. Detect column types

In [ ]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

In [ ]:
if "numeric_features" not in globals() or "categorical_features" not in globals():
    numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

column_type_summary = pd.DataFrame(
    {
        "column_type": ["numeric", "categorical"],
        "count": [len(numeric_features), len(categorical_features)],
        "columns": [numeric_features, categorical_features],
    }
)

display(column_type_summary)

## 9. Build preprocessing and random forest pipeline

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5)),
])

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2,
    )),
])


## 10. Train baseline random forest

In [ ]:
baseline_model.fit(X_train, y_train_log)

## 11. Predict and convert back to euro scale

In [ ]:
validation_pred_log = baseline_model.predict(X_validation)

In [ ]:
validation_pred = np.exp(validation_pred_log)

## 12. Preview Baseline Random Forest Predictions


In [ ]:
baseline_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred,
})

baseline_prediction_preview.head()

## 13. Baseline + Traficom features

This section tests whether external Traficom enrichment improves prediction compared to the listing-only baseline.

In [ ]:
features_baseline_plus_traficom = baseline_features + traficom_features

print("Number of baseline + Traficom features:", len(features_baseline_plus_traficom))
features_baseline_plus_traficom

## 14. Build X and y for baseline + Traficom

In [ ]:
missing_traficom_features = [
    column for column in features_baseline_plus_traficom if column not in train_df.columns
]
assert len(missing_traficom_features) == 0, (
    f"These selected features are missing from the dataset: {missing_traficom_features}"
)

X_train_traficom = train_df[features_baseline_plus_traficom].copy()
X_validation_traficom = validation_df[features_baseline_plus_traficom].copy()

## 15. Detect column types again

In [ ]:
numeric_features_traficom = X_train_traficom.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_traficom = X_train_traficom.select_dtypes(include=["object", "category"]).columns.tolist()

In [ ]:
print("Numeric columns:", numeric_features_traficom)
print("\nCategorical columns:", categorical_features_traficom)

## 16. Build a fresh preprocessing and random forest pipeline

In [ ]:
numeric_pipeline_traficom = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ])

categorical_pipeline_traficom = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5)),
])

In [ ]:
preprocessor_traficom = ColumnTransformer(transformers=[
    ("num", numeric_pipeline_traficom, numeric_features_traficom),
    ("cat", categorical_pipeline_traficom, categorical_features_traficom),
])

random_forest_traficom = Pipeline(steps=[
    ("preprocessor", preprocessor_traficom),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2,
    )),
])


## 17. Train baseline + Traficom random forest

In [ ]:
random_forest_traficom.fit(X_train_traficom, y_train_log)


## 18. Predict on validation and convert back to euro scale

In [ ]:
validation_pred_log_traficom = random_forest_traficom.predict(X_validation_traficom)


In [ ]:
validation_pred_traficom = np.exp(validation_pred_log_traficom)

## 19. Preview Baseline + Traficom Random Forest Predictions

In [ ]:
traficom_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred_traficom,
})

traficom_prediction_preview.head()

In [ ]:
traficom_prediction_preview.describe()

## 20. All Recommended Features

This section trains the random forest model on every recommended feature: the full manual feature union plus `first_seen_date`, `last_seen_date`, and `brand_is_known_model_family`, while still excluding IDs, duplicate keys, and the constant taxonomy flag.


In [ ]:
features_all = recommended_model_features

print("Number of recommended model features:", len(features_all))
features_all

## 21. Build X and y for all recommended features

In [ ]:
missing_all_features = [column for column in features_all if column not in train_df.columns]
assert len(missing_all_features) == 0, (
    f"These selected features are missing from the dataset: {missing_all_features}"
)

X_train_all = train_df[features_all].copy()
X_validation_all = validation_df[features_all].copy()

## 22. Detect column types again

In [ ]:
numeric_features_all = X_train_all.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_all = X_train_all.select_dtypes(include=["object", "category"]).columns.tolist()

In [ ]:
print("Numeric columns:", numeric_features_all)
print("\nCategorical columns:", categorical_features_all)

## 23. Build a fresh preprocessing and random forest pipeline

In [ ]:
numeric_pipeline_all = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ])

categorical_pipeline_all = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5)),
])

In [ ]:
preprocessor_all = ColumnTransformer(transformers=[
    ("num", numeric_pipeline_all, numeric_features_all),
    ("cat", categorical_pipeline_all, categorical_features_all),
])

random_forest_all = Pipeline(steps=[
    ("preprocessor", preprocessor_all),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2,
    )),
])


## 24. Train baseline + Traficom + registry lifecycle + listing dynamics random forest

In [ ]:
random_forest_all.fit(X_train_all, y_train_log)


## 25. Predict on validation and convert back to euro scale

In [ ]:
validation_pred_log_all = random_forest_all.predict(X_validation_all)


In [ ]:
validation_pred_all = np.exp(validation_pred_log_all)

## 26. Preview Recommended-Feature Predictions

These cells only preview the validation predictions in euro scale for the recommended-feature model. Formal model-comparison metrics and grouped diagnostics are reported in the later validation sections.

In [ ]:
# Pair each validation observation with its prediction in euro scale.
recommended_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred_all,
})

recommended_prediction_preview.head()

In [ ]:
# A quick descriptive summary helps check the prediction range before detailed validation.
recommended_prediction_preview.describe()

## 27. Validation Error Analysis By Part Group

This section evaluates validation-set errors for the existing random forest model by part grouping. It reuses the available validation predictions and reports which groups are predicted most accurately and least accurately.


In [ ]:
# Reuse the most detailed validation predictions already available in the notebook.
if "best_validation_predictions" in globals():
    validation_predictions_eur = best_validation_predictions
elif "validation_pred_all" in globals():
    validation_predictions_eur = validation_pred_all
elif "validation_pred_traficom" in globals():
    validation_predictions_eur = validation_pred_traficom
elif "validation_pred" in globals():
    validation_predictions_eur = validation_pred
else:
    raise NameError("No validation predictions were found in the notebook.")

# Build one validation results table in euro scale for grouped error analysis.
validation_results_df = validation_df[["price", "category", "subcategory", "part_name"]].copy()
validation_results_df = validation_results_df.rename(columns={"price": "actual_price"})
validation_results_df["predicted_price"] = validation_predictions_eur
validation_results_df["absolute_error"] = (
    validation_results_df["actual_price"]
    - validation_results_df["predicted_price"]
).abs()
validation_results_df["squared_error"] = (
    validation_results_df["actual_price"]
    - validation_results_df["predicted_price"]
) ** 2

validation_results_df.head()

In [ ]:
def summarize_group_errors(validation_results_df, group_column, min_count=20):
    summary = (
        validation_results_df
        .groupby(group_column, observed=False)
        .agg(
            count=("actual_price", "size"),
            MAE=("absolute_error", "mean"),
            median_absolute_error=("absolute_error", "median"),
            RMSE=("squared_error", lambda s: np.sqrt(s.mean())),
            within_10_eur=("absolute_error", lambda s: (s <= 10).mean()),
            within_20_eur=("absolute_error", lambda s: (s <= 20).mean()),
            within_50_eur=("absolute_error", lambda s: (s <= 50).mean()),
        )
        .reset_index()
    )

    summary = summary[summary["count"] >= min_count].copy()
    return summary.sort_values(["MAE", "RMSE", "count"]).reset_index(drop=True)


category_error_summary = summarize_group_errors(validation_results_df, "category")
subcategory_error_summary = summarize_group_errors(validation_results_df, "subcategory")
part_name_error_summary = summarize_group_errors(validation_results_df, "part_name")

## 28. Best And Worst Groups By MAE

The tables below summarize grouped validation errors in absolute euro terms. The `within_10_eur`, `within_20_eur`, and `within_50_eur` columns show how often predictions fall within practical absolute-error thresholds.

In [ ]:
# Show the main absolute-error metrics used for grouped validation review.
summary_display_columns = [
    "count",
    "MAE",
    "median_absolute_error",
    "RMSE",
    "within_10_eur",
    "within_20_eur",
    "within_50_eur",
]

for summary_name, summary_df, group_label in [
    ("Category", category_error_summary, "category"),
    ("Subcategory", subcategory_error_summary, "subcategory"),
    ("Part name", part_name_error_summary, "part_name"),
]:
    print(f"\n{summary_name}: 10 best groups by MAE")
    display(
        summary_df.head(10).style.format({
            "MAE": "{:.2f}",
            "median_absolute_error": "{:.2f}",
            "RMSE": "{:.2f}",
            "within_10_eur": "{:.1%}",
            "within_20_eur": "{:.1%}",
            "within_50_eur": "{:.1%}",
        })
    )

    print(f"{summary_name}: 10 worst groups by MAE")
    display(
        summary_df.sort_values(["MAE", "RMSE", "count"], ascending=[False, False, False]).head(10).style.format({
            "MAE": "{:.2f}",
            "median_absolute_error": "{:.2f}",
            "RMSE": "{:.2f}",
            "within_10_eur": "{:.1%}",
            "within_20_eur": "{:.1%}",
            "within_50_eur": "{:.1%}",
        })
    )


## 29. Select The Best Random Forest Feature Set

Train every candidate random forest configuration first, then validate them together once at the end so the feature-set comparison stays in one place.


In [ ]:
excluded_features = recommended_excluded_features

feature_sets = {
    "baseline only": baseline_features,
    "baseline + traficom_core": baseline_features + traficom_features,
    "baseline + traficom_core + registry_lifecycle": (
        baseline_features + traficom_features + registry_lifecycle_features
    ),
    "baseline + traficom_core + registry_lifecycle + traficom_extended": (
        baseline_features
        + traficom_features
        + registry_lifecycle_features
        + traficom_extended_features
    ),
    "previous manual all-features set": manual_all_feature_groups,
    "all recommended features": recommended_model_features,
}

experiment_results = []
experiment_feature_details = []
experiment_predictions = {}

In [ ]:
def prepare_experiment_features(feature_list, train_df, validation_df, excluded_features):
    requested_features = list(dict.fromkeys(feature_list))
    requested_features = [
        feature for feature in requested_features
        if feature not in excluded_features
    ]

    available_features = [
        feature for feature in requested_features
        if feature in train_df.columns
    ]
    missing_features = [
        feature for feature in requested_features
        if feature not in train_df.columns
    ]

    X_train_current = train_df[available_features].copy()
    X_validation_current = validation_df[available_features].copy()

    return (
        requested_features,
        available_features,
        missing_features,
        X_train_current,
        X_validation_current,
    )


def build_random_forest_pipeline(X_train_current):
    numeric_features_current = X_train_current.select_dtypes(
        include=["number", "bool"]
    ).columns.tolist()
    categorical_features_current = X_train_current.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    numeric_pipeline_current = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
            ])

    categorical_pipeline_current = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5)),
    ])

    preprocessor_current = ColumnTransformer(transformers=[
        ("num", numeric_pipeline_current, numeric_features_current),
        ("cat", categorical_pipeline_current, categorical_features_current),
    ])

    return Pipeline(steps=[
        ("preprocessor", preprocessor_current),
        ("model", RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
            min_samples_leaf=2,
        )),
    ])

In [ ]:
for model_name, feature_list in feature_sets.items():
    (
        requested_features,
        available_features,
        missing_features,
        X_train_current,
        X_validation_current,
    ) = prepare_experiment_features(
        feature_list,
        train_df,
        validation_df,
        excluded_features,
    )

    model_current = build_random_forest_pipeline(X_train_current)
    model_current.fit(X_train_current, y_train_log)

    validation_pred_log_current = model_current.predict(X_validation_current)
    validation_pred_current = np.exp(validation_pred_log_current)
    experiment_predictions[model_name] = validation_pred_current

    validation_mae_current = mean_absolute_error(
        y_validation,
        validation_pred_current,
    )
    validation_rmse_current = np.sqrt(
        mean_squared_error(y_validation, validation_pred_current)
    )
    validation_r2_current = r2_score(
        y_validation,
        validation_pred_current,
    )

    experiment_results.append({
        "experiment": model_name,
        "raw_columns_requested": len(requested_features),
        "raw_columns_used": len(available_features),
        "raw_columns_missing": len(missing_features),
        "validation_MAE": validation_mae_current,
        "validation_RMSE": validation_rmse_current,
        "validation_R2": validation_r2_current,
    })

    experiment_feature_details.append({
        "experiment": model_name,
        "used_features_count": len(available_features),
        "used_features_list": available_features,
        "missing_features_list": missing_features,
    })

## 30. Validate The Best Random Forest Model
 MAE as the main validation metrics, backup R^2 and MSE, and extra MAPE. 


In [ ]:
# Rebuild the best-model validation dataframe if this section is run on its own.
if "best_model_validation_df" not in globals():
    if "best_experiment_name" not in globals():
        if "experiment_results" not in globals() or len(experiment_results) == 0:
            raise NameError(
                "best_experiment_name is not defined. Run the feature-set comparison section first."
            )

        experiment_results_df = pd.DataFrame(experiment_results)
        best_experiment_name = experiment_results_df.sort_values(
            ["validation_MAE", "validation_RMSE"]
        ).iloc[0]["experiment"]

    best_validation_predictions = experiment_predictions[best_experiment_name]

    best_model_validation_df = pd.DataFrame({
        "actual_price": y_validation,
        "predicted_price": best_validation_predictions,
    })
    best_model_validation_df["residual"] = (
        best_model_validation_df["actual_price"]
        - best_model_validation_df["predicted_price"]
    )
    best_model_validation_df["absolute_error"] = (
        best_model_validation_df["residual"].abs()
    )
    best_model_validation_df["squared_error"] = (
        best_model_validation_df["residual"] ** 2
    )
    best_model_validation_df["absolute_percentage_error"] = (
        best_model_validation_df["absolute_error"]
        / best_model_validation_df["actual_price"].clip(lower=1)
    )
    best_model_validation_df["prediction_direction"] = np.where(
        best_model_validation_df["residual"] >= 0,
        "underpredicted",
        "overpredicted",
    )
    best_model_validation_df["actual_price_band"] = pd.qcut(
        best_model_validation_df["actual_price"],
        q=5,
        duplicates="drop",
    )

# Split the best-model validation errors by price band and prediction direction.
best_model_direction_summary = (
    best_model_validation_df
    .groupby(["actual_price_band", "prediction_direction"], observed=False)
    .agg(
        samples=("actual_price", "size"),
        mean_actual_price=("actual_price", "mean"),
        mean_predicted_price=("predicted_price", "mean"),
        mae=("absolute_error", "mean"),
    )
    .reset_index()
)

best_model_direction_summary.style.format({
    "mean_actual_price": "{:.1f}",
    "mean_predicted_price": "{:.1f}",
    "mae": "{:.2f}",
})

In [ ]:
# Build a full best-model validation table for absolute and percentage error diagnostics.
best_validation_predictions = experiment_predictions[best_experiment_name]

best_model_validation_df = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": best_validation_predictions,
})
best_model_validation_df["residual"] = (
    best_model_validation_df["actual_price"]
    - best_model_validation_df["predicted_price"]
)
best_model_validation_df["absolute_error"] = (
    best_model_validation_df["residual"].abs()
)
best_model_validation_df["squared_error"] = (
    best_model_validation_df["residual"] ** 2
)
best_model_validation_df["absolute_percentage_error"] = (
    best_model_validation_df["absolute_error"]
    / best_model_validation_df["actual_price"].clip(lower=1)
)
best_model_validation_df["prediction_direction"] = np.where(
    best_model_validation_df["residual"] >= 0,
    "underpredicted",
    "overpredicted",
)
best_model_validation_df["actual_price_band"] = pd.qcut(
    best_model_validation_df["actual_price"],
    q=5,
    duplicates="drop",
)

# The price-band summary includes MAPE because percentage error is informative across very different price levels.
best_model_price_band_summary = (
    best_model_validation_df
    .groupby("actual_price_band", observed=False)
    .agg(
        samples=("actual_price", "size"),
        min_actual_price=("actual_price", "min"),
        max_actual_price=("actual_price", "max"),
        mean_actual_price=("actual_price", "mean"),
        mae=("absolute_error", "mean"),
        rmse=("squared_error", lambda s: np.sqrt(s.mean())),
        mape=("absolute_percentage_error", "mean"),
        median_residual=("residual", "median"),
    )
    .reset_index()
)

best_model_price_band_summary.style.format({
    "min_actual_price": "{:.1f}",
    "max_actual_price": "{:.1f}",
    "mean_actual_price": "{:.1f}",
    "mae": "{:.2f}",
    "rmse": "{:.2f}",
    "mape": "{:.1%}",
    "median_residual": "{:+.2f}",
})

In [ ]:
experiment_results_df = pd.DataFrame(experiment_results)
baseline_row = experiment_results_df.loc[
    experiment_results_df["experiment"] == "baseline only"
].iloc[0]

experiment_results_df["delta_MAE_vs_baseline"] = (
    experiment_results_df["validation_MAE"]
    - baseline_row["validation_MAE"]
)
experiment_results_df["delta_RMSE_vs_baseline"] = (
    experiment_results_df["validation_RMSE"]
    - baseline_row["validation_RMSE"]
)
experiment_results_df["mae_rank"] = (
    experiment_results_df["validation_MAE"]
    .rank(method="dense")
    .astype(int)
)
experiment_results_df["rmse_rank"] = (
    experiment_results_df["validation_RMSE"]
    .rank(method="dense")
    .astype(int)
)

display_columns = [
    "experiment",
    "raw_columns_used",
    "validation_MAE",
    "delta_MAE_vs_baseline",
    "mae_rank",
    "validation_RMSE",
    "delta_RMSE_vs_baseline",
    "rmse_rank",
    "validation_R2",
]

best_experiment_name = experiment_results_df.sort_values(
    ["validation_MAE", "validation_RMSE"]
).iloc[0]["experiment"]


experiment_results_df.sort_values(
    ["validation_MAE", "validation_RMSE"]
)[display_columns].style.format({
    "validation_MAE": "{:.4f}",
    "delta_MAE_vs_baseline": "{:+.4f}",
    "validation_RMSE": "{:.4f}",
    "delta_RMSE_vs_baseline": "{:+.4f}",
    "validation_R2": "{:.4f}",
})